# XTraffic ST-GNN — Colab training (free T4 GPU)

CPU training of the full model is slow; a free Colab **T4** trains METR-LA to convergence comfortably. This notebook clones the repo, installs pinned deps, downloads/processes the data, and runs Phase 2 training.

**Runtime → Change runtime type → T4 GPU** before running.

Replace `REPO_URL` with your GitHub remote.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
REPO_URL = 'https://github.com/<your-username>/OlympiFlow.git'  # <-- EDIT ME
!git clone $REPO_URL
%cd OlympiFlow

In [ ]:
# Colab already has a recent torch. Install the rest (torch-geometric is a Phase 3
# dependency; Phase 2 training only needs torch + numpy + pandas + pyyaml + matplotlib).
!pip install -q pyyaml==6.0.1 requests==2.31.0 matplotlib scipy

In [ ]:
# Rebuild the processed tensors from scratch (reproducible; nothing manual).
!python -m xtraffic.data.pipelines.metr_la

In [ ]:
# Full training run. pick_device() auto-selects the T4 CUDA device.
!python -m xtraffic.models.gnn.train --config configs/train_metr_la.yaml

In [ ]:
# Baselines + final test-set evaluation of the best checkpoint.
!python -m xtraffic.models.gnn.baselines --config configs/train_metr_la.yaml
!python -m xtraffic.models.gnn.evaluate --checkpoint xtraffic/models/gnn/checkpoints/metr_la_best.pt --dataset metr_la

In [ ]:
# Download the trained checkpoint + logs back to your machine.
from google.colab import files
files.download('xtraffic/models/gnn/checkpoints/metr_la_best.pt')